# Homework 1 - Deep Learning Winter 2024

Student 1: Rajaa Haj 322512690

Student 2: Aseel Shaheen 212393532



In [8]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [58]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [59]:
import pandas as pd

# Define the paths to your datasets
train_path = '/content/drive/MyDrive/HW-1/part1_train.csv'
test_path = '/content/drive/MyDrive/HW-1/part1_test.csv'

# Load the datasets
train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

# Display the first few rows of the training dataset
# print(train_data.head())
# print(test_data.head())


# **In order to scale the data we need first to covnert non-numerical values into numerical represtnation. so what we do is we apply one-hot encoding in order to represnt them.**

In [96]:

# Identify categorical columns that exist in both train and test datasets
categorical_columns = list(set(train_data.select_dtypes(include=['object']).columns) &
                           set(test_data.select_dtypes(include=['object']).columns))

# Extract target variable before encoding (assuming last column is the target)
y_train = train_data.iloc[:, -1]
y_test = test_data.iloc[:, -1]

#need to remove the '.' in order to make the data clear and now convert it to numerical.
y_test = test_data.iloc[:, -1].str.strip().str.replace('.', '', regex=False)
# Strip whitespace from target column values
y_train = y_train.str.strip()
y_test = y_test.str.strip()


# Define the mapping for target values
label_mapping = {"<=50K": 0, ">50K": 1}

# Apply the mapping
y_train = y_train.map(label_mapping)
y_test = y_test.map(label_mapping)


# Apply one-hot encoding separately (for training and test (without target))
train_data_encoded = pd.get_dummies(train_data, columns=categorical_columns)
test_data_encoded = pd.get_dummies(test_data, columns=categorical_columns)

# Align columns between train and test datasets to handle mismatched categories
train_data_aligned, test_data_aligned = train_data_encoded.align(test_data_encoded, join='inner', axis=1)


# **Exctract features and exclude the target for training and test and scaling the data using the StandardScaler.**

In [97]:

# Extract features (after alignment)
X_train = train_data_aligned.drop(columns=train_data_aligned.columns[-1])  # Drop target column
X_test = test_data_aligned.drop(columns=test_data_aligned.columns[-1])    # Drop target column

# Normalize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# # Print shapes to verify
# print("X_train_scaled shape:", X_train_scaled.shape)
# print("X_test_scaled shape:", X_test_scaled.shape)
# print("y_train shape:", y_train.shape)
# print("y_test shape:", y_test.shape)


In [98]:
# Convert data to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

Now we Print the distribution of yearly income for the individuals and the
percentage of them more / less than 50k/year

In [100]:
# Compute distribution for y_train
unique, counts = torch.unique(y_train, return_counts=True)

# Print the distribution
print("Income Distribution in Training Data:")
for value, count in zip(unique, counts):
    label = "<=50K" if value.item() == 0 else ">50K"
    print(f"{label}: {count.item()}")

# Compute percentages
total = counts.sum().item()
percentages = (counts.float() / total) * 100

# Print the percentages
print("\nIncome Percentage in Training Data:")
for value, percent in zip(unique, percentages):
    label = "<=50K" if value.item() == 0 else ">50K"
    print(f"{label}: {percent.item():.2f}%")


Income Distribution in Training Data:
<=50K: 24719
>50K: 7841

Income Percentage in Training Data:
<=50K: 75.92%
>50K: 24.08%
